# Module 02: Pandas for Machine Learning
## Notebook 04: Grouping, Aggregations, and Pivot Tables

Aggregations allow data scientists to compress millions of fine-grained transactions into meaningful entity-level features (e.g., customer lifetime spend, average session duration, sensor anomaly rates). This notebook covers the famous Split-Apply-Combine pattern, multi-metric aggregations, and multidimensional pivot tables.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Apply the Split-Apply-Combine workflow using `.groupby()`.
2. Compute multi-metric aggregations across distinct columns with `.agg()`.
3. Normalize features within groups without changing row count using `.transform()`.
4. Reshape data into two-dimensional summary tables using `pd.pivot_table()`.
5. Compute contingency matrices for categorical variables using `pd.crosstab()`.
6. **Advanced:** Construct expanding windows, cumulative aggregations, within-group percentile rankings, and custom group `.apply()` functions.

In [ ]:
import pandas as pd
import numpy as np

# Generate realistic e-commerce transaction dataset
np.random.seed(42)
n_records = 15

transaction_data = {
    'Customer_ID': np.random.choice([101, 102, 103, 104, 105], size=n_records),
    'Category': np.random.choice(['Electronics', 'Books', 'Home'], size=n_records),
    'Order_Amount': np.round(np.random.uniform(15.0, 350.0, size=n_records), 2),
    'Discount_Code': np.random.choice(['PROMO10', 'WELCOME20', 'NONE'], size=n_records),
    'Delivery_Days': np.random.choice([1, 2, 3, 5, 7], size=n_records)
}

df_sales = pd.DataFrame(transaction_data)
print("E-commerce Transaction Dataset Preview (First 8 rows):")
print(df_sales.head(8))

---
### 1. The Split-Apply-Combine Pattern

The core concept of grouping:
1. **Split:** The DataFrame is partitioned into groups based on key columns.
2. **Apply:** A function (mean, sum, count, std) is computed independently on each group.
3. **Combine:** The individual results are concatenated into a resulting DataFrame.

In [ ]:
# Total spend and average spend per product category
category_summary = df_sales.groupby('Category')['Order_Amount'].mean().round(2)
print("Average Order Amount by Category:")
print(category_summary)

# Multi-column grouping: Customer spend per category
customer_category_spend = df_sales.groupby(['Customer_ID', 'Category'])['Order_Amount'].sum()
print("\nSpend per Customer per Category:")
print(customer_category_spend)

---
### 2. Multi-Metric Aggregations with `.agg()`

Machine learning feature tables often require multiple summary metrics per entity (e.g. min, max, mean, count, standard deviation).
Using `.agg()` with a dictionary allows granular, column-specific aggregations:

In [ ]:
customer_features = df_sales.groupby('Customer_ID').agg(
    Total_Spend=('Order_Amount', 'sum'),
    Mean_Order=('Order_Amount', 'mean'),
    Max_Order=('Order_Amount', 'max'),
    Order_Count=('Order_Amount', 'count'),
    Avg_Delivery=('Delivery_Days', 'mean')
).round(2)

print("Engineered Customer-Level Feature Table:")
print(customer_features)

---
### 3. Feature Engineering with `.transform()`

While `.agg()` collapses $N$ rows into $G$ group rows, `.transform()`:
- Computes group statistics, but **broadcasts the result back to match the original DataFrame length ($N$ rows)**.
- Perfect for computing relative features like:
$$\text{Relative Spend} = \frac{\text{Order Amount}}{\text{Category Mean}}$$

In [ ]:
# Compute category mean and attach directly to the original transactions
df_sales['Category_Mean_Amount'] = df_sales.groupby('Category')['Order_Amount'].transform('mean').round(2)

# Compute relative deviation feature
df_sales['Amount_Ratio_To_Category'] = (df_sales['Order_Amount'] / df_sales['Category_Mean_Amount']).round(2)

print("Transactions with Group-Transformed Features:")
print(df_sales[['Customer_ID', 'Category', 'Order_Amount', 'Category_Mean_Amount', 'Amount_Ratio_To_Category']].head(8))

---
### 4. Pivot Tables

`pd.pivot_table()` reshapes long data into wide cross-tabulated tables:
- `index`: Row groupings
- `columns`: Column groupings
- `values`: Numeric values to aggregate
- `aggfunc`: Aggregation function (`mean`, `sum`, `count`)
- `fill_value`: Value to replace missing combinations with

In [ ]:
pivot = pd.pivot_table(
    df_sales,
    index='Customer_ID',
    columns='Category',
    values='Order_Amount',
    aggfunc='sum',
    fill_value=0.0
)

print("Pivot Table: Total Spend by Customer across Categories:")
print(pivot)

---
### 5. Cross-Tabulations: `pd.crosstab()`

`pd.crosstab()` computes frequency contingency tables between categorical features, often used to evaluate categorical interactions or confusion matrices.

In [ ]:
# Frequency count of Discount Codes used across Categories
promo_by_cat = pd.crosstab(df_sales['Category'], df_sales['Discount_Code'], margins=True)
print("Contingency Table (Frequencies):")
print(promo_by_cat)

# Normalized cross-tabulation (Row percentages)
promo_row_pct = pd.crosstab(df_sales['Category'], df_sales['Discount_Code'], normalize='index').round(2) * 100
print("\nRow-Normalized Percentages (%):")
print(promo_row_pct)

---
### 6. Advanced Complex Usage: Cumulative Window Metrics, Within-Group Ranking, and Custom `.apply()`

In predictive modeling (especially fraud detection, customer retention, and algorithmic trading), aggregations must capture temporal trajectory and relative cohort position:
1. **Cumulative Metrics (`cumsum`, `cummax`):** Track historical running totals without data leakage.
2. **Within-Group Percentile Ranking (`rank(pct=True)`):** Determine where an observation ranks relative to its peers within the same segment.
3. **Custom Group `.apply()` returning multi-dimensional outputs:** Executes specialized logic across grouped sub-frames.

In [ ]:
# Sort transactions by customer and order amount
df_ordered = df_sales.sort_values(by=['Customer_ID', 'Order_Amount']).copy()

# 1. Cumulative spend trajectory per customer
df_ordered['Customer_Cumulative_Spend'] = df_ordered.groupby('Customer_ID')['Order_Amount'].cumsum()
df_ordered['Customer_Running_Max_Order'] = df_ordered.groupby('Customer_ID')['Order_Amount'].cummax()

# 2. Within-category percentile rank of order amount
df_ordered['Category_Spend_Percentile'] = df_ordered.groupby('Category')['Order_Amount'].rank(pct=True).round(3)

print("Transactions with Cumulative Metrics and Percentile Ranking:")
print(df_ordered[['Customer_ID', 'Category', 'Order_Amount', 'Customer_Cumulative_Spend', 'Category_Spend_Percentile']].head(10))

# 3. Custom Group .apply(): Extract top order and share of customer total spend
def summarize_customer_profile(group: pd.DataFrame) -> pd.Series:
    total_val = group['Order_Amount'].sum()
    top_order = group['Order_Amount'].max()
    top_share = top_order / total_val if total_val > 0 else 0.0
    return pd.Series({
        'Total_Spend': np.round(total_val, 2),
        'Largest_Order': np.round(top_order, 2),
        'Top_Order_Spend_Share': np.round(top_share, 3),
        'Unique_Categories': group['Category'].nunique()
    })

customer_profiles = df_sales.groupby('Customer_ID').apply(summarize_customer_profile, include_groups=False)
print("\nCustomer Deep Profiles via Custom Group .apply():")
print(customer_profiles)

### Summary & Next Steps
In this notebook, you mastered:
- The Split-Apply-Combine pattern with `.groupby()`.
- Granular multi-feature aggregations with `.agg()`.
- Feature engineering without aggregation collapse via `.transform()`.
- Multi-dimensional pivoting (`pivot_table`) and contingency matrices (`crosstab`).
- Cumulative tracking metrics (`cumsum`, `cummax`), within-group percentile ranking, and deep profiling with custom `.apply()`.

**Next Notebook:** `05_combining_datasets_and_timeseries.ipynb` — Relational joins, concatenation, datetime parsing, rolling windows, and as-of time series merging.